# 🧠 Notebook: Context Compression & Memory

In this notebook we look at what happens as an agent's conversation grows, and two families of tools for dealing with it: compressing what's in the context, and offloading information to memory outside of it.

## 📚 Sources

- [LangChain: Context Engineering](https://docs.langchain.com/oss/python/langchain/agents#context-management)
- [LangChain: Short-term Memory](https://docs.langchain.com/oss/python/langgraph/persistence)
- [LangChain: Long-term Memory](https://docs.langchain.com/oss/python/langchain/long-term-memory)

---

Good luck exploring context and memory! 🤗

## The Context Problem

Every model call sends the *entire* conversation so far - every message, every tool result. As an agent runs longer, that history keeps growing, which causes two separate problems:

1. **Hard limit:** every model has a fixed context window (a maximum number of tokens it can accept). Eventually, a long-running agent's history simply won't fit anymore.
2. **Soft costs:** even well before hitting that limit, a bigger context means slower, more expensive calls - and models tend to get worse at using information buried in the middle of a huge context ("lost in the middle").

There are two different families of solutions:

- **Context compression** - shrink what's already in the conversation (Sections 1-2 below).
- **Memory** - keep some information *outside* the conversation entirely, and bring it back only when it's actually needed (Sections 3-4 below).

We'll also look at **caching** (Section 5), which doesn't shrink context but avoids paying for the same call twice.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads LLM_HOST from a .env file in the project root (see notebook 03 / setup.md)

LLM_HOST = os.environ["LLM_HOST"]  # the IP address you got in the lecture
LLM_URL = f"http://{LLM_HOST}:11434"
LLM_REASONING = "gemma4:26b"  # the reasoning MoE model - also supports tool calling

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0, reasoning=False)

## 1. Context Compression: Summarization

`SummarizationMiddleware` watches the conversation, and once it crosses a `trigger` threshold (a number of messages, a token count, or a fraction of the context window), it replaces the older messages with an LLM-generated summary - keeping only the most recent messages (`keep`) intact. The summary is written back into the conversation itself, so it's a **permanent** rewrite of the history from that point on.

Let's set a very low trigger (6 messages) so we can see it kick in within a short demo conversation, and check that an important fact survives being summarized.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

summarizing_agent = create_agent(
    model=llm,
    tools=[],
    middleware=[SummarizationMiddleware(model=llm, trigger=("messages", 6), keep=("messages", 2))],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "summary-demo"}}
turns = [
    "My favorite color is blue.",
    "I have a dog named Rex.",
    "I live in Berlin.",
    "What is my favorite color?",
]
for turn in turns:
    result = summarizing_agent.invoke({"messages": [{"role": "user", "content": turn}]}, config=config)
    print(f"User: {turn}")
    print(f"Agent: {result['messages'][-1].content}")
    print(f"(messages currently in state: {len(result['messages'])})\n")

Watch the message count: it grows turn by turn, then drops once the trigger fires - the early messages about the dog and Berlin got compressed into a summary. Yet the agent still correctly answers "blue" at the end, because that fact made it into the summary. This is the tradeoff: you save context space, at the cost of the model only seeing a compressed version of anything older.

## 2. Context Compression: Clearing Old Tool Outputs

Summarization rewrites the whole conversation. `ContextEditingMiddleware` is more surgical: it targets large **tool outputs** specifically, since those are often the biggest single contributor to a bloated context (think: a huge search result you only needed once). Once the total size of tool outputs crosses a `trigger` (in tokens), older ones are replaced with a short placeholder in what gets *sent to the model* - keeping the most recent `keep` results intact.

Important distinction from Section 1: this only trims what's sent to the model on the *next* call. It does not rewrite the permanently stored transcript the way summarization does - so if you inspect `result["messages"]` afterwards, the full original tool outputs are still there. Only the model's-eye view shrinks.

In [ ]:
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit
from langchain.tools import tool


@tool
def get_book_summary(title: str) -> str:
    """Get a (very long, for demo purposes) summary for a book title."""
    return f"This is a very long summary of {title}. " * 20


editing_agent = create_agent(
    model=llm,
    tools=[get_book_summary],
    middleware=[ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=200, keep=1)])],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "edit-demo"}}
for title in ["Deep Learning", "Introduction to Algorithms", "Campbell Biology"]:
    result = editing_agent.invoke(
        {"messages": [{"role": "user", "content": f"Get me the summary of {title}"}]}, config=config
    )
    print(f"{title} -> {result['messages'][-1].content[:80]}...")

Even though each tool call returns a long, repeated block of text, the agent keeps responding quickly and doesn't spiral into resending every previous summary in full on every turn - `ClearToolUsesEdit` is quietly keeping the actual per-call context small behind the scenes.

## 3. Short-Term Memory: Remembering Within a Conversation

This is the `checkpointer` + `thread_id` pattern we've already used a few times. Without a checkpointer, every `.invoke()` call starts from a blank slate - the agent has no idea what happened in a previous call, even a moment ago. With a checkpointer, calls that share the same `thread_id` continue the *same* conversation; a different `thread_id` starts a brand new one, with no memory of the first.

In [ ]:
memory_agent = create_agent(model=llm, tools=[], checkpointer=InMemorySaver())

thread_a = {"configurable": {"thread_id": "conversation-a"}}
memory_agent.invoke({"messages": [{"role": "user", "content": "My name is Nils."}]}, config=thread_a)
result = memory_agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=thread_a)
print("Same thread:", result["messages"][-1].content)

thread_b = {"configurable": {"thread_id": "conversation-b"}}
result = memory_agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=thread_b)
print("Different thread:", result["messages"][-1].content)

## 4. Long-Term Memory: Remembering Across Conversations

Short-term memory only lasts for one `thread_id`. **Long-term memory** is for facts that should persist *everywhere* - a user's stated preferences, for example, which should still be known the next time they start a completely new conversation. LangGraph's `store` (e.g. `InMemoryStore`) is a simple key-value store for exactly this, shared across every thread.

To use it from a tool, add a parameter annotated with `ToolRuntime` - it's automatically injected (not something the model fills in), and gives the tool access to `runtime.store`.

In [ ]:
from langchain.tools import ToolRuntime
from langgraph.store.memory import InMemoryStore


@tool
def remember_preference(key: str, value: str, runtime: ToolRuntime) -> str:
    """Save a user preference for later, across all future conversations."""
    runtime.store.put(("preferences",), key, {"value": value})
    return f"Saved: {key} = {value}"


@tool
def recall_preference(key: str, runtime: ToolRuntime) -> str:
    """Recall a previously saved user preference."""
    item = runtime.store.get(("preferences",), key)
    if item is None:
        return f"No preference found for {key}"
    return f"{key} = {item.value['value']}"


store = InMemoryStore()
store_agent = create_agent(
    model=llm,
    tools=[remember_preference, recall_preference],
    checkpointer=InMemorySaver(),
    store=store,
)

session_1 = {"configurable": {"thread_id": "session-1"}}
result = store_agent.invoke(
    {"messages": [{"role": "user", "content": "Please remember that my favorite programming language is Python."}]},
    config=session_1,
)
print("Session 1:", result["messages"][-1].content)

session_2 = {"configurable": {"thread_id": "session-2"}}  # a brand new thread
result = store_agent.invoke(
    {"messages": [{"role": "user", "content": "What is my favorite programming language?"}]},
    config=session_2,
)
print("Session 2 (new thread):", result["messages"][-1].content)

Notice `session_2` used a completely different `thread_id` - by Section 3's rules, it should know nothing about `session_1`. But it still answers correctly, because `remember_preference` wrote to the shared `store`, not to the (per-thread) checkpointer. That's the whole distinction between the two kinds of memory: **short-term memory lives on the thread; long-term memory lives on the store.**

## 5. Caching

If the exact same prompt is sent to the model twice, there's no need to actually run it twice - the second call can be served from a cache instantly. This is mostly a development-time and testing convenience (rerunning a notebook cell, retrying a test suite) rather than something that helps end users, since it only fires on an *exact* match: even a one-character difference in the prompt is a cache miss, and if you want varied/creative answers (e.g. via `temperature`), caching identical inputs will just give you the same cached output every time, defeating the point.

`langchain_core`'s `InMemoryCache` attaches to a model directly. Let's compare the timing of the same call, run twice.

In [ ]:
import time
from langchain_core.caches import InMemoryCache

cached_llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0, reasoning=False, cache=InMemoryCache())

t0 = time.time()
cached_llm.invoke("What is the capital of France?")
print(f"First call:  {time.time() - t0:.2f}s (actually hit the model)")

t0 = time.time()
cached_llm.invoke("What is the capital of France?")
print(f"Second call: {time.time() - t0:.2f}s (served from cache)")

A quick note on naming: `create_agent` also has its own `cache=` parameter, but that one is a *different* thing - a LangGraph-level cache for skipping repeated graph node executions, not for LLM calls specifically. What we just used, an `InMemoryCache` attached directly to the model, is the one that caches actual model calls.

## Exercise: Short-Term + Long-Term Together

Build an agent that:

1. Has `remember_preference` and `recall_preference` as tools, a `checkpointer`, and a `store`.
2. In one thread, tell it your name and ask it to remember your favorite book (long-term, via the tool).
3. Still in the *same* thread, ask what your name is - it should know, from short-term memory, without needing a tool.
4. In a *new* thread, ask for your favorite book - it should still know, from long-term memory.
5. In that same new thread, ask for your name - it should **not** know this time.

In [ ]:
# Your code here...

<details>
<summary><b>Show solution</b></summary>

```python
exercise_store = InMemoryStore()
exercise_agent = create_agent(
    model=llm,
    tools=[remember_preference, recall_preference],
    checkpointer=InMemorySaver(),
    store=exercise_store,
)

thread_1 = {"configurable": {"thread_id": "exercise-1"}}
exercise_agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Nils. Please remember that my favorite book is Dune."}]},
    config=thread_1,
)

result = exercise_agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=thread_1)
print("Same thread, name:", result["messages"][-1].content)  # should know - short-term memory

thread_2 = {"configurable": {"thread_id": "exercise-2"}}
result = exercise_agent.invoke({"messages": [{"role": "user", "content": "What is my favorite book?"}]}, config=thread_2)
print("New thread, book:", result["messages"][-1].content)  # should know - long-term memory

result = exercise_agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=thread_2)
print("New thread, name:", result["messages"][-1].content)  # should NOT know - never stored long-term
```

</details>